# 模块概述

WtUftCore 是 WonderTrader UFT（Ultra Fast Trading，极速交易）策略运行的核心模块，负责提供UFT策略的实时交易环境。主要包括：
- UFT策略引擎和策略上下文管理
- 实时市场数据处理和分发
- 交易适配和订单管理
- 行情解析器适配
- 策略参数共享管理
- 事件通知机制
- 数据结构和辅助工具类

1. **数据定义层**（UftDataDefs）：
   - 定义UFT策略使用的核心数据结构
   - 包括持仓明细、订单、成交、回合等数据结构
   - 采用块头+数据数组的结构，支持内存映射文件存储
   - 是整个模块的数据基础

2. **引擎层**（WtUftEngine + WtUftTicker）：
   - WtUftEngine：UFT引擎核心，管理策略上下文、数据订阅、时间管理等
   - WtUftTicker：实时ticker，处理实时行情并触发分钟线闭合事件
   - 是整个框架的控制中枢

3. **策略层**（UftStrategyMgr + UftStraContext）：
   - UftStrategyMgr：策略管理器，动态加载策略工厂，创建策略实例
   - UftStraContext：策略上下文，管理策略的交易上下文、持仓、订单、数据订阅等
   - 使用UftDataDefs定义的数据结构存储持仓、订单、成交等数据
   - 提供策略运行环境和交易接口

4. **数据层**（WtUftDtMgr）：
   - 管理实时tick、历史tick、K线等市场数据
   - 实现IDataManager接口，提供统一的数据查询接口
   - 支持数据订阅和数据切片查询

5. **适配器层**（TraderAdapter + ParserAdapter）：
   - TraderAdapter：交易适配器，适配不同的交易接口，提供统一的交易操作
   - ParserAdapter：行情解析器适配器，适配不同的行情数据源，统一行情接口

6. **工具支持层**（EventNotifier + ActionPolicyMgr + ShareManager + WtHelper）：
   - EventNotifier：事件通知器，通过消息队列异步广播交易事件
   - ActionPolicyMgr：动作策略管理器，管理交易动作的执行规则
   - ShareManager：共享内存管理器，管理策略参数的共享和持久化
   - WtHelper：辅助工具类，提供路径管理和时间管理功能


# 层次关系图
```mermaid
graph LR
    %% 样式定义
    classDef engineClass fill:#e1f5ff,stroke:#01579b,stroke-width:3px,color:#000;
    classDef contextClass fill:#fff3e0,stroke:#e65100,stroke-width:2px,color:#000;
    classDef adapterClass fill:#f3e5f5,stroke:#4a148c,stroke-width:2px,color:#000;
    classDef dataClass fill:#e8f5e9,stroke:#1b5e20,stroke-width:2px,color:#000;
    classDef utilClass fill:#fce4ec,stroke:#880e4f,stroke-width:2px,color:#000;
    classDef tickerClass fill:#e0f2f1,stroke:#004d40,stroke-width:2px,color:#000;
    classDef mgrClass fill:#ffe0b2,stroke:#e65100,stroke-width:2px,color:#000;
    classDef interfaceClass fill:#f5f5f5,stroke:#616161,stroke-width:1px,stroke-dasharray: 5 5,color:#000;
    classDef defClass fill:#fffde7,stroke:#f57f17,stroke-width:2px,color:#000;

    %% 接口层
    subgraph Interfaces["接口层 - 抽象接口"]
        direction TB
        IUftStraCtx["IUftStraCtx<br/>UFT策略上下文接口<br/>• 交易接口<br/>• 数据查询接口<br/>• 持仓管理接口"]
        ITrdNotifySink["ITrdNotifySink<br/>交易通知接口<br/>• 成交回报<br/>• 订单回报<br/>• 持仓更新<br/>• 通道状态"]
        IParserStub["IParserStub<br/>行情解析器存根接口<br/>• Tick推送<br/>• 订单队列推送<br/>• 订单明细推送<br/>• 成交明细推送"]
        IDataManager["IDataManager<br/>数据管理器接口<br/>• Tick切片查询<br/>• K线切片查询<br/>• Level-2数据查询"]
        ITraderSpi["ITraderSpi<br/>交易接口回调<br/>• 连接回调<br/>• 登录回调<br/>• 订单回调<br/>• 成交回调"]
        IParserSpi["IParserSpi<br/>行情解析器回调<br/>• Tick回调<br/>• 订单队列回调<br/>• 订单明细回调<br/>• 成交明细回调"]
    end

    %% 数据定义层
    subgraph DataDefs["数据定义层 - 数据结构"]
        direction TB
        UftDataDefs["UftDataDefs<br/>UFT数据定义<br/>• BlockHeader 数据块头<br/>• DetailStruct 持仓明细<br/>• PositionBlock 持仓块<br/>• OrderStruct 订单结构<br/>• OrderBlock 订单块<br/>• TradeStruct 成交结构<br/>• TradeBlock 成交块<br/>• RoundStruct 回合结构<br/>• RoundBlock 回合块"]:::defClass
    end

    %% 引擎层
    subgraph Engines["引擎层 - 核心控制"]
        direction TB
        WtUftEngine["WtUftEngine<br/>UFT引擎<br/>• 策略上下文管理<br/>• 数据订阅管理<br/>• 数据分发<br/>• 时间管理<br/>• 交易日管理"]:::engineClass
        WtUftTicker["WtUftTicker<br/>实时Ticker<br/>• 实时行情处理<br/>• 分钟线闭合判断<br/>• 交易日判断<br/>• 后台定时检查"]:::tickerClass
    end

    %% 策略管理层
    subgraph StrategyMgrs["策略管理层 - 策略生命周期"]
        direction TB
        UftStrategyMgr["UftStrategyMgr<br/>策略管理器<br/>• 加载策略工厂<br/>• 创建策略实例<br/>• 管理策略映射"]:::mgrClass
        UftStraContext["UftStraContext<br/>策略上下文<br/>• 交易接口实现<br/>• 本地持仓管理<br/>• 订单管理<br/>• 数据订阅管理<br/>• 事件转发"]:::contextClass
    end

    %% 数据管理层
    subgraph DataLayer["数据管理层 - 市场数据"]
        direction TB
        WtUftDtMgr["WtUftDtMgr<br/>数据管理器<br/>• 实时Tick缓存<br/>• 历史Tick缓存<br/>• K线缓存<br/>• 数据切片查询"]:::dataClass
    end

    %% 适配器层
    subgraph Adapters["适配器层 - 外部接口适配"]
        direction TB
        TraderAdapter["TraderAdapter<br/>交易适配器<br/>• 交易接口适配<br/>• 订单管理<br/>• 持仓管理<br/>• 动作策略转换<br/>• 风险控制"]:::adapterClass
        ParserAdapter["ParserAdapter<br/>行情解析器适配器<br/>• 行情接口适配<br/>• 数据过滤<br/>• 数据标准化<br/>• 数据转发"]:::adapterClass
    end

    %% 工具支持层
    subgraph Utils["工具支持层 - 辅助功能"]
        direction TB
        EventNotifier["EventNotifier<br/>事件通知器<br/>• 消息队列集成<br/>• 异步事件处理<br/>• JSON格式转换<br/>• 事件广播"]:::utilClass
        ActionPolicyMgr["ActionPolicyMgr<br/>动作策略管理器<br/>• 交易动作规则<br/>• 品种规则映射<br/>• 手数限制管理"]:::utilClass
        ShareManager["ShareManager<br/>共享内存管理器<br/>• 参数共享域管理<br/>• 参数读写<br/>• 参数监控<br/>• 参数同步"]:::utilClass
        WtHelper["WtHelper<br/>辅助工具类<br/>• 路径管理<br/>• 时间管理<br/>• 目录创建"]:::utilClass
    end

    %% 继承关系
    WtUftEngine -.->|"实现"| IParserStub
    UftStraContext -.->|"实现"| IUftStraCtx
    UftStraContext -.->|"实现"| ITrdNotifySink
    WtUftDtMgr -.->|"实现"| IDataManager
    TraderAdapter -.->|"实现"| ITraderSpi
    ParserAdapter -.->|"实现"| IParserSpi

    %% 核心组合关系
    WtUftEngine -->|"包含"| WtUftTicker
    WtUftEngine -->|"管理"| UftStraContext
    WtUftEngine -->|"使用"| WtUftDtMgr
    WtUftEngine -->|"使用"| EventNotifier
    
    UftStrategyMgr -->|"创建"| UftStraContext
    UftStraContext -->|"使用"| TraderAdapter
    UftStraContext -->|"使用"| ShareManager
    UftStraContext -->|"使用数据结构"| UftDataDefs
    
    WtUftTicker -->|"触发事件"| WtUftEngine
    
    %% 数据流关系
    ParserAdapter -->|"推送行情"| WtUftEngine
    WtUftEngine -->|"分发数据"| UftStraContext
    WtUftEngine -->|"更新数据"| WtUftDtMgr
    
    %% 交易流关系
    UftStraContext -->|"下单"| TraderAdapter
    TraderAdapter -->|"交易回报"| UftStraContext
    TraderAdapter -->|"使用"| ActionPolicyMgr
    TraderAdapter -->|"通知事件"| EventNotifier
    
    %% 工具层关系
    WtUftEngine -.->|"使用"| WtHelper
    UftStraContext -.->|"使用"| WtHelper
    WtUftDtMgr -.->|"使用"| WtHelper
    TraderAdapter -.->|"使用"| WtHelper
    ParserAdapter -.->|"使用"| WtHelper
    
    %% 应用样式
    class WtUftEngine engineClass
    class UftStraContext contextClass
    class TraderAdapter,ParserAdapter adapterClass
    class WtUftDtMgr dataClass
    class EventNotifier,ActionPolicyMgr,ShareManager,WtHelper utilClass
    class WtUftTicker tickerClass
    class UftStrategyMgr mgrClass
    class IUftStraCtx,ITrdNotifySink,IParserStub,IDataManager,ITraderSpi,IParserSpi interfaceClass
    class UftDataDefs defClass
```


# 适配器层

## ParserAdapter.h/cpp — 行情解析器适配器

### 框架图
```mermaid
graph LR
    %% 样式定义
    classDef parserInterface fill:#e3f2fd,stroke:#1565c0,stroke-width:2px,color:#000;
    classDef parserAdapter fill:#fff3e0,stroke:#e65100,stroke-width:3px,color:#000;
    classDef parserMgr fill:#f3e5f5,stroke:#6a1b9a,stroke-width:2px,color:#000;
    classDef execInterface fill:#e8f5e9,stroke:#2e7d32,stroke-width:2px,color:#000;
    classDef execUnit fill:#fff9c4,stroke:#f57f17,stroke-width:3px,color:#000;
    classDef execFactory fill:#fce4ec,stroke:#c2185b,stroke-width:2px,color:#000;
    classDef external fill:#f5f5f5,stroke:#616161,stroke-width:1px,stroke-dasharray: 5 5,color:#000;

    %% =======================
    %% 外部接口层
    %% =======================
    subgraph ExternalLayer["外部接口层"]
        direction TB
        IParserApi["IParserApi<br/>行情解析器API<br/>数据源接口"]
        IParserSpi["IParserSpi<br/>行情解析器回调接口<br/>接收数据回调"]
        WtUftEngine["WtUftEngine<br/>UFT引擎<br/>数据接收者"]:::external
    end

    %% =======================
    %% 行情解析适配器层
    %% =======================
    subgraph ParserLayer["行情解析适配器层 - ParserAdapter"]
        direction TB
        IParserStub["IParserStub<br/>数据推送接口<br/>定义数据接收接口"]:::parserInterface
        ParserAdapter["ParserAdapter<br/>行情解析器适配器<br/>适配不同数据源<br/>数据过滤与转发"]:::parserAdapter
        ParserAdapterMgr["ParserAdapterMgr<br/>适配器管理器<br/>管理多个适配器"]:::parserMgr
    end

    %% =======================
    %% 继承关系（虚线）
    %% =======================
    ParserAdapter -.->|"实现"| IParserSpi
    WtUftEngine -.->|"实现"| IParserStub

    %% =======================
    %% 组合与管理关系（实线）
    %% =======================
    ParserAdapterMgr -->|"管理"| ParserAdapter
    ParserAdapter -->|"使用"| IParserApi
    ParserAdapter -->|"推送数据"| IParserStub
    IParserStub -->|"数据接收者"| WtUftEngine


    %% =======================
    %% 数据流关系
    %% =======================
    IParserApi -->|"数据回调"| ParserAdapter
    ParserAdapter -->|"过滤转发"| WtUftEngine

    %% 应用样式
    class IParserStub parserInterface
    class ParserAdapter parserAdapter
    class ParserAdapterMgr parserMgr
    class ExecuteContext execInterface
    class ExecuteUnit execUnit
    class IExecuterFact execFactory
    class IParserApi,IParserSpi,WtUftEngine external
```

## TraderAdapter.h/cpp — 交易适配器

### 框架图
```mermaid
graph LR
    %% 样式定义
    classDef traderInterface fill:#e3f2fd,stroke:#1565c0,stroke-width:2px,color:#000;
    classDef traderAdapter fill:#fff3e0,stroke:#e65100,stroke-width:3px,color:#000;
    classDef traderMgr fill:#f3e5f5,stroke:#6a1b9a,stroke-width:2px,color:#000;
    classDef notifyInterface fill:#e8f5e9,stroke:#2e7d32,stroke-width:2px,color:#000;
    classDef policyMgr fill:#fce4ec,stroke:#c2185b,stroke-width:2px,color:#000;
    classDef context fill:#fff9c4,stroke:#f57f17,stroke-width:2px,color:#000;
    classDef external fill:#f5f5f5,stroke:#616161,stroke-width:1px,stroke-dasharray: 5 5,color:#000;

    %% =======================
    %% 外部接口层
    %% =======================
    subgraph ExternalLayer["外部接口层"]
        direction TB
        ITraderApi["ITraderApi<br/>交易接口API<br/>交易通道接口"]
        ITraderSpi["ITraderSpi<br/>交易接口回调接口<br/>接收交易回调"]
        UftStraContext["UftStraContext<br/>策略上下文<br/>交易请求者"]:::context
    end

    %% =======================
    %% 交易适配器层
    %% =======================
    subgraph TraderLayer["交易适配器层 - TraderAdapter"]
        direction TB
        ITrdNotifySink["ITrdNotifySink<br/>交易通知接口<br/>定义交易事件接收接口"]:::notifyInterface
        TraderAdapter["TraderAdapter<br/>交易适配器<br/>适配不同交易接口<br/>订单与持仓管理<br/>风险控制"]:::traderAdapter
        TraderAdapterMgr["TraderAdapterMgr<br/>适配器管理器<br/>管理多个适配器"]:::traderMgr
    end

    %% =======================
    %% 工具支持层
    %% =======================
    subgraph UtilsLayer["工具支持层"]
        direction TB
        ActionPolicyMgr["ActionPolicyMgr<br/>动作策略管理器<br/>交易动作规则管理"]:::policyMgr
    end

    %% =======================
    %% 继承关系（虚线）
    %% =======================
    TraderAdapter -.->|"实现"| ITraderSpi
    UftStraContext -.->|"实现"| ITrdNotifySink

    %% =======================
    %% 组合与管理关系（实线）
    %% =======================
    TraderAdapterMgr -->|"管理"| TraderAdapter
    TraderAdapter -->|"使用"| ITraderApi
    TraderAdapter -->|"使用"| ActionPolicyMgr
    TraderAdapter -->|"通知"| ITrdNotifySink
    ITrdNotifySink -->|"数据接收者"| UftStraContext

    %% =======================
    %% 交易流关系
    %% =======================
    UftStraContext -->|"下单请求"| TraderAdapter
    ITraderApi -->|"交易回调"| TraderAdapter
    TraderAdapter -->|"交易回报"| UftStraContext

    %% 应用样式
    class ITrdNotifySink notifyInterface
    class TraderAdapter traderAdapter
    class TraderAdapterMgr traderMgr
    class ActionPolicyMgr policyMgr
    class UftStraContext context
    class ITraderApi,ITraderSpi external
```

# 策略管理层

## 框架图
```mermaid
graph LR
    %% 样式定义
    classDef dataStruct fill:#e3f2fd,stroke:#1565c0,stroke-width:2px,color:#000;
    classDef dataBlock fill:#bbdefb,stroke:#0d47a1,stroke-width:2px,color:#000;
    classDef context fill:#fff3e0,stroke:#e65100,stroke-width:3px,color:#000;
    classDef wrapper fill:#f3e5f5,stroke:#6a1b9a,stroke-width:2px,color:#000;
    classDef manager fill:#e8f5e9,stroke:#2e7d32,stroke-width:2px,color:#000;
    classDef interface fill:#f5f5f5,stroke:#616161,stroke-width:1px,stroke-dasharray: 5 5,color:#000;
    classDef strategy fill:#fff9c4,stroke:#f57f17,stroke-width:2px,color:#000;

    %% =======================
    %% 数据定义层 - UftDataDefs.h
    %% =======================
    subgraph DataLayer["数据定义层 - UftDataDefs.h"]
        direction TB
        
        subgraph BaseStruct["基础结构"]
            BlockHeader["BlockHeader<br/>数据块头<br/>标志/类型/日期/容量/大小"]:::dataStruct
        end
        
        subgraph DataStructs["数据结构"]
            DetailStruct["DetailStruct<br/>持仓明细结构<br/>交易所/合约/方向/数量/价格/盈亏"]:::dataStruct
            OrderStruct["OrderStruct<br/>订单结构<br/>交易所/合约/方向/开平/数量/价格/状态"]:::dataStruct
            TradeStruct["TradeStruct<br/>成交结构<br/>交易所/合约/方向/开平/数量/价格/时间"]:::dataStruct
            RoundStruct["RoundStruct<br/>回合结构<br/>交易所/合约/方向/开平价格/盈亏"]:::dataStruct
        end
        
        subgraph DataBlocks["数据块"]
            PositionBlock["PositionBlock<br/>持仓数据块<br/>继承BlockHeader<br/>包含DetailStruct数组"]:::dataBlock
            OrderBlock["OrderBlock<br/>订单数据块<br/>继承BlockHeader<br/>包含OrderStruct数组"]:::dataBlock
            TradeBlock["TradeBlock<br/>成交数据块<br/>继承BlockHeader<br/>包含TradeStruct数组"]:::dataBlock
            RoundBlock["RoundBlock<br/>回合数据块<br/>继承BlockHeader<br/>包含RoundStruct数组"]:::dataBlock
        end
    end

    %% =======================
    %% 策略上下文层 - UftStraContext.h/cpp
    %% =======================
    subgraph ContextLayer["策略上下文层 - UftStraContext"]
        direction TB
        
        subgraph ContextClass["上下文类"]
            UftStraContext["UftStraContext<br/>策略上下文<br/>实现IUftStraCtx和ITrdNotifySink<br/>管理持仓/订单/成交/回合<br/>数据持久化与事件转发"]:::context
        end
        
        subgraph BlockPairs["数据块配对"]
            PosBlkPair["PosBlkPair<br/>持仓数据块配对<br/>PositionBlock + 内存映射文件 + 锁"]:::dataBlock
            OrdBlkPair["OrdBlkPair<br/>订单数据块配对<br/>OrderBlock + 内存映射文件 + 锁"]:::dataBlock
            TrdBlkPair["TrdBlkPair<br/>成交数据块配对<br/>TradeBlock + 内存映射文件 + 锁"]:::dataBlock
            RndBlkPair["RndBlkPair<br/>回合数据块配对<br/>RoundBlock + 内存映射文件 + 锁"]:::dataBlock
        end
    end

    %% =======================
    %% 策略管理层 - UftStrategyMgr.h/cpp
    %% =======================
    subgraph ManagerLayer["策略管理层 - UftStrategyMgr"]
        direction TB
        
        subgraph WrapperClass["包装器类"]
            UftStraWrapper["UftStraWrapper<br/>策略包装器<br/>包装策略实例和工厂<br/>管理策略生命周期"]:::wrapper
        end
        
        subgraph ManagerClass["管理器类"]
            UftStrategyMgr["UftStrategyMgr<br/>策略管理器<br/>加载策略工厂<br/>创建和管理策略实例"]:::manager
        end
    end

    %% =======================
    %% 外部接口层
    %% =======================
    subgraph InterfaceLayer["外部接口层"]
        direction TB
        IUftStraCtx["IUftStraCtx<br/>策略上下文接口<br/>交易接口/数据查询/持仓管理"]:::interface
        ITrdNotifySink["ITrdNotifySink<br/>交易通知接口<br/>成交回报/订单回报/持仓更新"]:::interface
        UftStrategy["UftStrategy<br/>策略基类<br/>定义策略生命周期回调"]:::strategy
        IUftStrategyFact["IUftStrategyFact<br/>策略工厂接口<br/>创建/删除策略实例"]:::interface
    end

    %% =======================
    %% 继承关系（虚线）
    %% =======================
    PositionBlock -.->|"继承"| BlockHeader
    OrderBlock -.->|"继承"| BlockHeader
    TradeBlock -.->|"继承"| BlockHeader
    RoundBlock -.->|"继承"| BlockHeader
    
    UftStraContext -.->|"实现"| IUftStraCtx
    UftStraContext -.->|"实现"| ITrdNotifySink
    UftStrategy -.->|"由工厂创建"| IUftStrategyFact

    %% =======================
    %% 组合关系（实线）
    %% =======================
    PositionBlock -->|"包含"| DetailStruct
    OrderBlock -->|"包含"| OrderStruct
    TradeBlock -->|"包含"| TradeStruct
    RoundBlock -->|"包含"| RoundStruct
    
    PosBlkPair -->|"使用"| PositionBlock
    OrdBlkPair -->|"使用"| OrderBlock
    TrdBlkPair -->|"使用"| TradeBlock
    RndBlkPair -->|"使用"| RoundBlock
    
    UftStraContext -->|"管理"| PosBlkPair
    UftStraContext -->|"管理"| OrdBlkPair
    UftStraContext -->|"管理"| TrdBlkPair
    UftStraContext -->|"管理"| RndBlkPair
    UftStraContext -->|"绑定"| UftStrategy
    
    UftStraWrapper -->|"包装"| UftStrategy
    UftStraWrapper -->|"持有"| IUftStrategyFact
    UftStrategyMgr -->|"管理"| UftStraWrapper
    UftStrategyMgr -->|"加载"| IUftStrategyFact

    %% =======================
    %% 数据流关系
    %% =======================
    UftStrategy -->|"使用"| UftStraContext
    UftStraContext -->|"事件转发"| UftStrategy
    UftStrategyMgr -->|"创建策略"| UftStraWrapper

    %% 应用样式
    class BlockHeader,DetailStruct,OrderStruct,TradeStruct,RoundStruct dataStruct
    class PositionBlock,OrderBlock,TradeBlock,RoundBlock,PosBlkPair,OrdBlkPair,TrdBlkPair,RndBlkPair dataBlock
    class UftStraContext context
    class UftStraWrapper wrapper
    class UftStrategyMgr manager
    class IUftStraCtx,ITrdNotifySink,IUftStrategyFact interface
    class UftStrategy strategy
```

## UftDataDefs.h — 数据定义

## UftStraContext.h/cpp — 策略上下文类

## UftStrategyMgr.h/cpp — 策略管理器

# 数据管理器 WtUftDtMgr.h/cpp

## 框架图

```mermaid
graph LR
    %% 样式定义
    classDef interface fill:#e3f2fd,stroke:#1565c0,stroke-width:2px,color:#000;
    classDef manager fill:#fff3e0,stroke:#e65100,stroke-width:3px,color:#000;
    classDef cache fill:#e8f5e9,stroke:#2e7d32,stroke-width:2px,color:#000;
    classDef notify fill:#fce4ec,stroke:#c2185b,stroke-width:2px,color:#000;
    classDef external fill:#f5f5f5,stroke:#616161,stroke-width:1px,stroke-dasharray: 5 5,color:#000;

    %% =======================
    %% 接口层
    %% =======================
    subgraph InterfaceLayer["接口层"]
        direction TB
        IDataManager["IDataManager<br/>数据管理器接口<br/>定义数据查询接口"]:::interface
    end

    %% =======================
    %% 数据管理器层 - WtUftDtMgr.h/cpp
    %% =======================
    subgraph ManagerLayer["数据管理器层 - WtUftDtMgr"]
        direction TB
        
        subgraph ManagerClass["管理器类"]
            WtUftDtMgr["WtUftDtMgr<br/>UFT数据管理器<br/>实现IDataManager接口<br/>• 实时行情处理<br/>• 数据缓存管理<br/>• 数据查询接口<br/>• 数据订阅管理"]:::manager
        end
        
        subgraph CacheLayer["数据缓存层"]
            RtTickMap["_rt_tick_map<br/>实时Tick缓存<br/>存储最新Tick数据"]:::cache
            TicksCache["_ticks_cache<br/>历史Tick缓存<br/>存储历史Tick数据"]:::cache
            BarsCache["_bars_cache<br/>K线缓存<br/>存储K线数据"]:::cache
        end
        
        subgraph Subscription["订阅管理"]
            SubedBasicBars["_subed_basic_bars<br/>已订阅基础K线集合"]:::cache
        end
        
        subgraph NotifyStruct["通知结构"]
            NotifyItem["NotifyItem<br/>K线通知项"]:::notify
            BarNotifies["_bar_notifies<br/>K线通知项列表"]:::notify
        end
    end

    %% =======================
    %% 外部依赖层
    %% =======================
    subgraph ExternalLayer["外部依赖层"]
        direction TB
        WtUftEngine["WtUftEngine<br/>UFT引擎<br/>数据管理器使用者"]:::external
        WTSDataFactory["WTSDataFactory<br/>数据工厂<br/>创建数据对象"]:::external
        WTSVariant["WTSVariant<br/>配置变体类<br/>配置参数"]:::external
    end

    %% =======================
    %% 继承关系（虚线）
    %% =======================
    WtUftDtMgr -.->|"实现"| IDataManager

    %% =======================
    %% 组合关系（实线）
    %% =======================
    WtUftDtMgr -->|"管理"| RtTickMap
    WtUftDtMgr -->|"管理"| TicksCache
    WtUftDtMgr -->|"管理"| BarsCache
    WtUftDtMgr -->|"管理"| SubedBasicBars
    WtUftDtMgr -->|"管理"| BarNotifies
    BarNotifies -->|"包含"| NotifyItem

    %% =======================
    %% 数据流关系
    %% =======================
    WtUftEngine -->|"使用"| WtUftDtMgr
    WtUftEngine -->|"推送行情"| WtUftDtMgr
    WTSDataFactory -->|"创建数据"| WtUftDtMgr
    WTSVariant -->|"配置"| WtUftDtMgr

    %% 应用样式
    class IDataManager interface
    class WtUftDtMgr manager
    class RtTickMap,TicksCache,BarsCache,SubedBasicBars cache
    class WtUftEngine,WTSDataFactory,WTSVariant external
    class NotifyItem,BarNotifies notify
```

# 引擎层

## 框架图

```mermaid
graph LR
    %% 样式定义
    classDef interface fill:#e3f2fd,stroke:#1565c0,stroke-width:2px,color:#000;
    classDef engine fill:#fff3e0,stroke:#e65100,stroke-width:3px,color:#000;
    classDef ticker fill:#e8f5e9,stroke:#2e7d32,stroke-width:2px,color:#000;
    classDef subscription fill:#f3e5f5,stroke:#6a1b9a,stroke-width:2px,color:#000;
    classDef context fill:#fff9c4,stroke:#f57f17,stroke-width:2px,color:#000;
    classDef time fill:#e0f2f1,stroke:#004d40,stroke-width:2px,color:#000;
    classDef external fill:#f5f5f5,stroke:#616161,stroke-width:1px,stroke-dasharray: 5 5,color:#000;

    %% =======================
    %% 接口层
    %% =======================
    subgraph InterfaceLayer["接口层"]
        direction TB
        IParserStub["IParserStub<br/>行情解析器存根接口<br/>定义数据推送接口"]:::interface
    end

    %% =======================
    %% 引擎层 - WtUftEngine.h/cpp
    %% =======================
    subgraph EngineLayer["引擎层 - WtUftEngine"]
        direction TB
        
        subgraph EngineClass["引擎类"]
            WtUftEngine["WtUftEngine<br/>UFT引擎<br/>实现IParserStub接口<br/>• 策略上下文管理<br/>• 数据订阅管理<br/>• 数据分发<br/>• 时间管理<br/>• 交易日管理"]:::engine
        end
        
        subgraph SubscriptionLayer["订阅管理层"]
            TickSubMap["_tick_sub_map<br/>Tick订阅映射表<br/>合约代码 → 策略上下文ID集合"]:::subscription
            OrdQueSubMap["_ordque_sub_map<br/>订单队列订阅映射表<br/>合约代码 → 策略上下文ID集合"]:::subscription
            OrdDtlSubMap["_orddtl_sub_map<br/>订单明细订阅映射表<br/>合约代码 → 策略上下文ID集合"]:::subscription
            TransSubMap["_trans_sub_map<br/>成交明细订阅映射表<br/>合约代码 → 策略上下文ID集合"]:::subscription
            BarSubMap["_bar_sub_map<br/>K线订阅映射表<br/>合约代码-周期-倍数 → 策略上下文ID集合"]:::subscription
        end
        
        subgraph ContextLayer["上下文管理层"]
            ContextMap["_ctx_map<br/>策略上下文映射表<br/>策略上下文ID → 策略上下文指针"]:::context
        end
        
        subgraph TimeLayer["时间管理层"]
            CurDate["_cur_date<br/>当前日期<br/>YYYYMMDD格式"]:::time
            CurTime["_cur_time<br/>当前时间<br/>HHMMSS格式"]:::time
            CurRawTime["_cur_raw_time<br/>原始时间<br/>HHMMSS格式"]:::time
            CurSecs["_cur_secs<br/>当前秒数<br/>包含毫秒"]:::time
            CurTDate["_cur_tdate<br/>当前交易日<br/>YYYYMMDD格式"]:::time
        end
    end

    %% =======================
    %% Ticker层 - WtUftTicker.h/cpp
    %% =======================
    subgraph TickerLayer["Ticker层 - WtUftTicker"]
        direction TB
        
        subgraph TickerClass["Ticker类"]
            WtUftRtTicker["WtUftRtTicker<br/>实时Ticker<br/>• 实时行情处理<br/>• 分钟线闭合判断<br/>• 交易日判断<br/>• 后台定时检查"]:::ticker
        end
        
        subgraph TickerTimeMgr["Ticker时间管理"]
            TickerDate["_date<br/>当前日期"]:::time
            TickerTimeVal["_time<br/>当前时间"]:::time
            CurPos["_cur_pos<br/>当前分钟位置<br/>交易时段内的分钟数"]:::time
            NextCheckTime["_next_check_time<br/>下次检查时间<br/>时间戳（毫秒）"]:::time
            LastEmitPos["_last_emit_pos<br/>上次触发分钟位置"]:::time
        end
        
        subgraph TickerThreadMgr["线程管理"]
            Stopped["_stopped<br/>停止标志"]:::time
            Thread["_thrd<br/>后台线程指针"]:::time
            Mutex["_mtx<br/>互斥锁<br/>保护共享数据"]:::time
        end
    end

    %% =======================
    %% 外部依赖层
    %% =======================
    subgraph ExternalLayer["外部依赖层"]
        direction TB
        WtUftDtMgr["WtUftDtMgr<br/>数据管理器<br/>市场数据管理"]:::external
        TraderAdapterMgr["TraderAdapterMgr<br/>交易适配器管理器<br/>交易接口管理"]:::external
        EventNotifier["EventNotifier<br/>事件通知器<br/>事件广播"]:::external
        IBaseDataMgr["IBaseDataMgr<br/>基础数据管理器<br/>合约/商品信息查询"]:::external
        WTSSessionInfo["WTSSessionInfo<br/>交易时段信息<br/>交易时间模板"]:::external
        UftStraContext["UftStraContext<br/>策略上下文<br/>策略运行环境"]:::external
    end

    %% =======================
    %% 继承关系（虚线）
    %% =======================
    WtUftEngine -.->|"实现"| IParserStub

    %% =======================
    %% 组合关系（实线）
    %% =======================
    WtUftEngine -->|"包含"| WtUftRtTicker
    WtUftEngine -->|"管理"| TickSubMap
    WtUftEngine -->|"管理"| OrdQueSubMap
    WtUftEngine -->|"管理"| OrdDtlSubMap
    WtUftEngine -->|"管理"| TransSubMap
    WtUftEngine -->|"管理"| BarSubMap
    WtUftEngine -->|"管理"| ContextMap
    WtUftEngine -->|"管理"| CurDate
    WtUftEngine -->|"管理"| CurTime
    WtUftEngine -->|"管理"| CurRawTime
    WtUftEngine -->|"管理"| CurSecs
    WtUftEngine -->|"管理"| CurTDate
    
    WtUftRtTicker -->|"持有"| WtUftEngine
    WtUftRtTicker -->|"管理"| TickerDate
    WtUftRtTicker -->|"管理"| TickerTimeVal
    WtUftRtTicker -->|"管理"| CurPos
    WtUftRtTicker -->|"管理"| NextCheckTime
    WtUftRtTicker -->|"管理"| LastEmitPos
    WtUftRtTicker -->|"管理"| Stopped
    WtUftRtTicker -->|"管理"| Thread
    WtUftRtTicker -->|"管理"| Mutex
    WtUftRtTicker -->|"使用"| WTSSessionInfo

    %% =======================
    %% 数据流关系
    %% =======================
    IParserStub -->|"数据推送"| WtUftEngine
    WtUftEngine -->|"转发Tick"| WtUftRtTicker
    WtUftRtTicker -->|"触发分钟闭合"| WtUftEngine
    WtUftRtTicker -->|"触发交易日事件"| WtUftEngine
    WtUftEngine -->|"分发数据"| TickSubMap
    WtUftEngine -->|"分发数据"| OrdQueSubMap
    WtUftEngine -->|"分发数据"| OrdDtlSubMap
    WtUftEngine -->|"分发数据"| TransSubMap
    WtUftEngine -->|"分发数据"| BarSubMap
    TickSubMap -->|"订阅关系"| ContextMap
    OrdQueSubMap -->|"订阅关系"| ContextMap
    OrdDtlSubMap -->|"订阅关系"| ContextMap
    TransSubMap -->|"订阅关系"| ContextMap
    BarSubMap -->|"订阅关系"| ContextMap
    ContextMap -->|"管理"| UftStraContext
    
    WtUftEngine -->|"使用"| WtUftDtMgr
    WtUftEngine -->|"使用"| TraderAdapterMgr
    WtUftEngine -->|"使用"| EventNotifier
    WtUftEngine -->|"使用"| IBaseDataMgr
    WtUftRtTicker -->|"查询"| IBaseDataMgr

    %% 应用样式
    class IParserStub interface
    class WtUftEngine engine
    class WtUftRtTicker ticker
    class TickSubMap,OrdQueSubMap,OrdDtlSubMap,TransSubMap,BarSubMap subscription
    class ContextMap context
    class CurDate,CurTime,CurRawTime,CurSecs,CurTDate,TickerDate,TickerTimeVal,CurPos,NextCheckTime,LastEmitPos,Stopped,Thread,Mutex time
    class WtUftDtMgr,TraderAdapterMgr,EventNotifier,IBaseDataMgr,WTSSessionInfo,UftStraContext external
```

## WtUftTicker.h/cpp — UFT实时ticker

## WtUftEngine.h/cpp — UFT引擎

# 接口层

## ITrdNotifySink.h/cpp — 交易通知接口

# 工具支持层

## ActionPolicyMgr.h/cpp — 动作策略管理器

## EventNotifier.h/cpp — 事件通知器

## ShareManager.h/cpp — 共享内存管理器

## WtHelper.h/cpp — 辅助工具类